# 🔬 XRD Analysis of Starch Polymorphs
### FBMFOR — Food Fraud Analysis | University of Reading

This notebook guides you through:
1. Loading XRD diffractogram data from the course dataset
2. Visualising patterns as a **waterfall plot**
3. Identifying and assigning **polymorph peaks** (A-type or B-type)
4. Estimating **relative crystallinity**
5. Identifying an **unknown starch sample** by peak matching

---
> **Fraud context:** Starch is a common adulterant in spices, cocoa, and dairy powders.  
> XRD polymorph fingerprinting can identify starch botanical origin and detect substitution.

## 1 · Setup

In [ ]:
# ============================================================
# SETUP — INSTALL AND IMPORT LIBRARIES
# ============================================================
# All libraries used here are pre-installed in Google Colab.
# If running locally, install with:  pip install numpy pandas matplotlib scipy

# --- Core numerical and data libraries ---
import numpy as np                       # Array operations and numerical computation
import pandas as pd                      # DataFrames for tabular data handling

# --- Plotting ---
import matplotlib.pyplot as plt          # Static publication-quality figures
import matplotlib.patches as mpatches    # Custom legend patches (coloured rectangles)
from matplotlib.lines import Line2D      # Custom legend line handles

# --- Signal processing ---
from scipy.signal import find_peaks, savgol_filter
# find_peaks   : local-maxima detection algorithm (used in Sections 6 & 8)
# savgol_filter: Savitzky-Golay smoothing (used in Section 4)

from scipy.interpolate import interp1d   # 1D interpolation (used when resampling spectra)
import os, io                            # File-system and in-memory byte-stream utilities

# --- Colab file upload helper ---
# Imported inside a try/except so the notebook also works in a local Jupyter session.
# IN_COLAB is used throughout to switch between upload and local-file code paths.
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("\u2713 Libraries loaded")
print(f"  Running in Google Colab: {IN_COLAB}")

## 2 · Reference Peak Library

Literature 2θ positions (Cu Kα, λ = 1.5406 Å) for the main starch polymorphs.
Edit this cell if you want to add new starch types or adjust positions.

In [ ]:
# ============================================================
# REFERENCE PEAK LIBRARY
# ============================================================
# Literature 2θ positions (Cu Kα, λ = 1.5406 Å) for the two starch
# polymorph families present in the dataset.
#
# The crystal structure of starch is determined by the double-helix
# packing of amylopectin chains in the semicrystalline lamellae:
#
#   A-type (monoclinic unit cell, space group B2):
#     Characteristic of cereal starches (Wheat, Corn, Rice).
#     The double helices are packed more tightly and with less water
#     than B-type. Diagnostic peaks appear at 15.0°, 17.0°, 17.9°, 
#     22.9°.
#
#   B-type (hexagonal unit cell, space group P6₁):
#     Characteristic of tuber and root starches (Potato, Tapioca).
#     Wider channels in the lattice accommodate more water molecules.
#     The hallmark low-angle peak at ~5.6° arises from the larger
#     d-spacing of the (100) reflection in the hexagonal lattice.
#     5.6, 15.1, 17.2, 19.7, 22.2, 24.0, 26.4
#
# Source: Dome K, Podgorbunskikh E, Bychkov A, Lomovsky O. 
# Changes in the Crystallinity Degree of Starch Having Different 
# Types of Crystal Structure after Mechanical Pretreatment. 
# Polymers (Basel). 2020 Mar 12;12(3):641. 
# doi: 10.3390/polym12030641. PMID: 32178224; PMCID: PMC7183072.

REFERENCE_PEAKS = {
    # A-type (cereal starches) — monoclinic unit cell
    # Peaks at 15.3°/17.1°/18.0° form the characteristic A-type triplet.
    "Wheat":   {"peaks": [15.0, 17.0, 17.9, 22.9],             "polymorph": "A-type"},
    "Corn":    {"peaks": [15.0, 17.0, 17.9, 22.9],             "polymorph": "A-type"},

    # B-type (tuber / root starches) — hexagonal unit cell
    # The 5.6° peak is the most diagnostic: it is absent in A-type starches.
    "Potato":  {"peaks": [5.6, 15.1, 17.2, 19.7, 22.2, 24.0, 26.4], "polymorph": "B-type"},
    "Tapioca": {"peaks": [5.6, 15.1, 17.2, 19.7, 22.2, 24.0, 26.4], "polymorph": "B-type"},
}

# --- Colour maps ---
# One colour per polymorph family, used for reference-peak markers and the legend.
POLYMORPH_COLOURS = {
    "A-type":  "#C0392B",   # Red family — cereal starches
    "B-type":  "#1F618D",   # Blue family — tuber/root starches
}

# One colour per individual starch, used for diffractogram lines and fill.
# A-type starches share red tones; B-type starches share blue tones;
# Unknown is purple to remain visually neutral until identified.
STARCH_COLOURS = {
    "Wheat":   "#922B21",   # Dark red
    "Corn":    "#E74C3C",   # Bright red
    "Potato":  "#1A5276",   # Dark blue
    "Tapioca": "#2E86C1",   # Bright blue
    "Unknown": "#6C3483",   # Purple (polymorph unknown a priori)
}

print("\u2713 Reference library loaded")
for name, info in REFERENCE_PEAKS.items():
    print(f"  {name:12s} ({info['polymorph']:7s})  peaks: {info['peaks']}")

## 3 · Load Data

Data are loaded directly from the course GitHub repository.
If you are running this notebook locally with `data/xrd.csv` present, it will be used instead.

The expected format is a merged CSV with columns `TwoTheta, Counts, Starch`.
All five starch samples (Corn, Potato, Tapioca, Wheat, Unknown) are in a single file.

In [ ]:
# ============================================================
# LOAD XRD DATA
# ============================================================
# The dataset (xrd.csv) contains diffractograms for five starch types:
#   Corn, Potato, Tapioca, Wheat, Unknown
# Each diffractogram was measured from 5° to 65° 2θ using Cu Kα radiation.
# The file is a long-format CSV with three columns:
#   TwoTheta  — 2θ angle in degrees
#   Counts    — raw detector counts (arbitrary intensity units)
#   Starch    — sample label
#
# Loading is attempted in order of convenience:
#   1. GitHub raw URL  — works in Colab with no extra steps
#   2. Colab file upload — fallback if GitHub is unreachable
#   3. Local file path  — for running the notebook on your own machine

GITHUB_URL = (
    "https://raw.githubusercontent.com/ggkuhnle/fbmfor-foodfraud/main/data/xrd.csv"
)

df_raw = None  # Will hold the raw DataFrame once loaded

# --- 1: Try GitHub URL ---
# pandas.read_csv() accepts a URL directly. This is the simplest route in Colab
# as it requires no file upload and always fetches the current course dataset.
try:
    df_raw = pd.read_csv(GITHUB_URL)
    print(f"\u2713 Loaded data from GitHub ({df_raw.shape[0]} rows)")
except Exception as e:
    print(f"  GitHub load failed ({e}); trying upload/local …")

# --- 2: Colab file upload ---
# If the GitHub fetch failed (e.g., network restricted), offer a manual upload.
if df_raw is None and IN_COLAB:
    print("\U0001f4c2 Upload xrd.csv:")
    uploaded = colab_files.upload()
    fname = list(uploaded.keys())[0]
    df_raw = pd.read_csv(io.BytesIO(uploaded[fname]))
    print(f"\u2713 Loaded from upload ({df_raw.shape[0]} rows)")

# --- 3: Local file fallback ---
# When running in Jupyter outside Colab, look for the file in the standard
# repository layout (data/xrd.csv) or the current working directory.
if df_raw is None:
    local_paths = ["data/xrd.csv", "xrd.csv"]
    for p in local_paths:
        if os.path.exists(p):
            df_raw = pd.read_csv(p)
            print(f"\u2713 Loaded from {p} ({df_raw.shape[0]} rows)")
            break

if df_raw is None:
    raise FileNotFoundError(
        "Could not load xrd.csv. Check your internet connection or upload the file."
    )

# --- Validate and parse ---
# Strip any accidental whitespace from column names (common in exported CSVs).
df_raw.columns = [c.strip() for c in df_raw.columns]
required = {"TwoTheta", "Counts", "Starch"}
if not required.issubset(df_raw.columns):
    raise ValueError(f"Missing columns. Found: {list(df_raw.columns)}, need: {required}")

# Build spectra_dict: { starch_label: (two_theta_array, counts_array) }
# groupby() splits the long-format DataFrame by sample label.
# sort_values() ensures 2θ runs in ascending order for all subsequent analysis.
spectra_dict = {}
for starch, grp in df_raw.groupby("Starch"):
    grp = grp.sort_values("TwoTheta")
    spectra_dict[starch.strip()] = (grp["TwoTheta"].values, grp["Counts"].values)
    print(f"  {starch}: {len(grp)} points, "
          f"2θ {grp['TwoTheta'].min():.1f}–{grp['TwoTheta'].max():.1f}°")

print(f"\n\u2713 {len(spectra_dict)} starch pattern(s): {list(spectra_dict.keys())}")

## 4 · Preprocessing

Optional smoothing (Savitzky–Golay filter).  
Set `SMOOTH = False` to skip.

In [ ]:
# ============================================================
# PREPROCESSING — SMOOTHING AND NORMALISATION
# ============================================================
# Two lightweight preprocessing steps prepare the raw counts for analysis:
#
#   1. Savitzky-Golay (SG) smoothing
#      Fits a low-degree polynomial to a sliding window of points and replaces
#      the central value with the fitted value. This reduces Poisson counting
#      noise without distorting peak positions or widths, provided the window
#      is narrow relative to the peak width.
#
#
#   2. Min-max normalisation to 0-100
#      Scales each diffractogram independently so its minimum maps to 0 and its
#      maximum maps to 100. This allows direct visual comparison of patterns
#      regardless of differences in raw counting statistics between samples
#      (e.g., due to different packing density in the XRD sample holder).

# ── PARAMETERS — modify and re-run to experiment ───────────────────────────────────
SMOOTH       = True    # Set to False to skip smoothing entirely.
SG_WINDOW    = 11      # Window length in data points (must be odd).
                       # At a typical step size of 0.02°/point, 11 points ≈ 0.22°.
                       # Increase (e.g., to 21) for noisier data;
                       # decrease if peaks appear rounded or flattened.
SG_POLYORDER = 3       # Polynomial degree. 3 gives good peak shape preservation.
                       # Must be less than SG_WINDOW.
# ──────────────────────────────────────────────────────────────────────────────


def normalise(counts):
    """
    Min-max normalise a 1D intensity array to the range [0, 100].

    Scaling to 0-100 rather than 0-1 keeps values at a human-readable scale
    when printed or displayed in the waterfall plot.

    Parameters:
        counts : 1D numpy array — raw or smoothed detector counts

    Returns:
        1D numpy array — rescaled intensities in [0, 100]
    """
    lo, hi = counts.min(), counts.max()
    if hi > lo:
        return (counts - lo) / (hi - lo) * 100
    else:
        return np.zeros_like(counts)  # Flat pattern: return zeros rather than NaN


# --- Apply smoothing and normalisation to every sample ---
# processed stores a tuple (two_theta, smoothed_counts, normalised_counts).
# Both the raw-scale smoothed counts (used later for crystallinity integration)
# and the 0-100 normalised version (used for plotting and peak detection)
# are retained so that each downstream step can use the appropriate scale.
processed = {}
for starch, (tt, ct) in spectra_dict.items():
    if SMOOTH:
        # Guard against the window exceeding the array length (rare with real data,
        # but important for robustness with very short test arrays).
        window = min(SG_WINDOW, len(ct) - 1)
        window = window if window % 2 == 1 else window - 1  # Enforce odd length
        ct_smooth = savgol_filter(ct, window_length=window, polyorder=SG_POLYORDER)
        ct_smooth = np.maximum(ct_smooth, 0)  # Clip negative artefacts near zero baseline
    else:
        ct_smooth = ct.copy()
    processed[starch] = (tt, ct_smooth, normalise(ct_smooth))

print(f"\u2713 Preprocessing complete ({'SG smoothed' if SMOOTH else 'no smoothing'})")
print(f"  Samples: {list(processed.keys())}")

## 5 · Waterfall Plot

Each diffractogram is offset vertically for clarity.  
Peak reference positions are annotated per starch type.

In [ ]:
# ============================================================
# WATERFALL PLOT
# ============================================================
# A waterfall plot stacks each diffractogram vertically, offset by a fixed
# amount, so all patterns can be compared side-by-side without overlap.
# This is the standard way to present a series of powder diffractograms.
#
# Two visual aids help pattern recognition:
#   - Dashed vertical grid lines mark the diagnostic 2θ positions for both
#     polymorph families, making it easy to see which samples share peaks.
#   - Small arrows mark the expected reference peak positions for each labelled
#     starch. The Unknown sample has no reference arrows because its identity
#     is what we are trying to determine.

# ── PARAMETERS — modify and re-run ────────────────────────────────────────────────
X_MIN       = 5.0     # Lower 2θ limit for the plot (degrees).
                      # 5° captures the B-type low-angle peak at 5.6°.
X_MAX       = 40.0    # Upper 2θ limit. All diagnostic starch peaks fall below 25°;
                      # extending to 40° confirms the absence of unexpected reflections.
OFFSET_STEP = 115     # Vertical offset between patterns (normalised intensity units).
                      # Increase if patterns overlap; decrease for a more compact view.
SHOW_GRID   = True    # Show dashed vertical lines at diagnostic 2θ positions.
SAVE_FIGURE = True    # Save the figure as a PNG file.
FIGURE_DPI  = 200     # Resolution in dots per inch (150 for screen, 300 for print).
FIGURE_NAME = "xrd_waterfall.png"
# ──────────────────────────────────────────────────────────────────────────────

starch_order = list(processed.keys())
n = len(starch_order)

fig, ax = plt.subplots(figsize=(12, 2.2 * n + 1.5))
ax.set_facecolor("white")

# --- Diagnostic grid lines ---
# Draw thin dashed verticals at the reference peak positions for both
# polymorph types. Spanning all patterns lets students see at a glance
# which samples produce a reflection at each diagnostic position.
if SHOW_GRID:
    for gp in [5.6, 14.4, 15.3, 17.1, 17.2, 18.0, 19.8, 22.2, 23.0, 24.0]:
        if X_MIN <= gp <= X_MAX:
            ax.axvline(gp, color="grey", lw=0.5, ls="--", alpha=0.35, zorder=0)

# --- Draw each diffractogram ---
for i, starch in enumerate(starch_order):
    tt, ct_raw, ct_norm = processed[starch]

    # Restrict to the plotted 2θ window
    mask = (tt >= X_MIN) & (tt <= X_MAX)
    tt_m, ct_m = tt[mask], ct_norm[mask]

    base = i * OFFSET_STEP   # Baseline elevation for this pattern
    y    = ct_m + base        # Shift the pattern upward by the cumulative offset

    col = STARCH_COLOURS.get(starch, "#555555")

    # Semi-transparent fill under the curve (aids visual separation of patterns)
    ax.fill_between(tt_m, base, y, alpha=0.18, color=col, zorder=i + 1)
    ax.plot(tt_m, y, color=col, lw=1.5, zorder=i + 2)  # Solid pattern line
    ax.axhline(base, color="grey", lw=0.4, ls="-", alpha=0.5, zorder=0)  # Baseline rule

    # --- Sample label in the right margin ---
    # Show the polymorph type for reference starches; leave it blank for Unknown.
    poly = REFERENCE_PEAKS.get(starch, {}).get("polymorph", "")
    label_text = f"{starch}\n({poly})" if poly else starch
    ax.text(X_MAX + 0.5, base + 48,
            label_text, fontsize=9, color=col, fontweight="bold",
            va="center", ha="left", zorder=i + 3)

    # --- Reference peak arrows ---
    # Draw upward-pointing arrows at the expected 2θ positions for each reference
    # starch. Unknown is not in REFERENCE_PEAKS so no arrows appear for it,
    # prompting students to read the peak positions directly from the pattern.
    ref = REFERENCE_PEAKS.get(starch)
    if ref:
        pcol = POLYMORPH_COLOURS[ref["polymorph"]]
        for p in ref["peaks"]:
            if X_MIN <= p <= X_MAX:
                idx = np.argmin(np.abs(tt_m - p))  # Nearest data point to reference
                py  = y[idx]
                ax.annotate("", xy=(p, py + 3), xytext=(p, py + 13),
                            arrowprops=dict(arrowstyle="-|>", color=pcol,
                                            lw=1.1, mutation_scale=7),
                            zorder=i + 4)
                ax.text(p, py + 14, f"{p}°",
                        fontsize=5.5, color=pcol, fontweight="bold",
                        ha="center", va="bottom", rotation=90, zorder=i + 5)

# --- Axes and labels ---
ax.set_xlim(X_MIN, X_MAX + 8)   # Extra space on right for sample labels
ax.set_ylim(-15, n * OFFSET_STEP + 30)
ax.set_xlabel(r"2$\theta$ / degrees (Cu K$\alpha$)", fontsize=12)
ax.set_yticks([])                # Suppress y-axis: absolute intensity is not meaningful
ax.set_title("XRD Diffractograms of Starch Types", fontsize=14,
             fontweight="bold", pad=10)
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# --- Legend ---
# Build patches only for polymorph types actually present in the dataset.
# The set comprehension avoids a KeyError if POLYMORPH_COLOURS does not
# contain an entry for a polymorph not in the reference library.
present_polymorphs = {REFERENCE_PEAKS.get(s, {}).get("polymorph")
                      for s in starch_order} - {None}
patches = [mpatches.Patch(fc=POLYMORPH_COLOURS[p], label=p)
           for p in ["A-type", "B-type"]
           if p in present_polymorphs]
ax.legend(handles=patches, title="Polymorph", title_fontsize=9,
          fontsize=8.5, loc="upper left", framealpha=0.85)

ax.text(0.5, -0.06,
        "Reference positions shown (\u25b2). Dashed lines: diagnostic 2\u03b8. Region 5–40\u00b0.",
        transform=ax.transAxes, ha="center", fontsize=8, color="grey")

plt.tight_layout()
if SAVE_FIGURE:
    plt.savefig(FIGURE_NAME, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"\u2713 Figure saved as {FIGURE_NAME}")
plt.show()

### What to look for in the waterfall plot

**B-type polymorphs** (Potato, Tapioca) show a distinctive peak at low angle (~5–6°).
This arises from the wider spacing of the double-helix network in the hexagonal B-type lattice.

**A-type polymorphs** (Wheat, Corn) lack this low-angle peak and instead show a tight cluster
of three peaks between 15° and 18°, plus a strong peak near 23°.
These arise from the more compact monoclinic A-type unit cell.


